# Jupyter Notebook for AIFSv2 Single

This notebook is for running the AIFSv2 Single model on google colab. It works for the environment versions in the following table. Check if your python Version is 3.12.x for the precompiled flash-attn binaries.
| Runtime Version |Python Version| Supported | GPUs Tested |
|---|---|---|---|
| Latest | 3.13.15 | ❌ |  |
| 2026.07 | 3.12.13 | ✅ |L4|


In [ ]:
!python --version

# Installation

Install the matching versions of PyTorch, Torchvision, Torchaudio, TorchCodec, and supporting geospatial libraries required for this project.

In [ ]:
!pip install -q anemoi-inference[huggingface]==0.8.3 anemoi-models==0.9.3 anemoi-utils==0.4.35.post3
!pip install -q --upgrade torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 torchcodec==0.9.0 --index-url https://download.pytorch.org/whl/cu128 
!pip install -q torch-geometric==2.8.0 
!pip install -q earthkit-regrid==0.5.1 ecmwf-opendata==0.3.29 "earthkit-data<1.0.0" "earthkit-utils>=1.0,<2.0" earthkit.geo
!pip install -q zarr cartopy
!pip install -q flash-attn==2.8.3

# Import packages

In [ ]:
import platform
import torch
import flash_attn

import datetime
from collections import defaultdict
from pathlib import Path
import os
import requests
import copy

import numpy as np

import earthkit.data as ekd
import earthkit.regrid as ekr
from earthkit.geo.grids.array import regrid

import xarray as xr

from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from ecmwf.opendata import Client as OpendataClient

from google.colab import drive
import shutil

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Check Runtime

In [ ]:

print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available. This notebook requires a GPU.")

# See https://github.com/huggingface/transformers/issues/28188
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)

print("GPU:", gpu_name)
print(f"Compute capability: {major}.{minor}")

if major < 8:
    raise RuntimeError(
        "FlashAttention requires Ampere GPUs or newer"
    )

print("FlashAttention:", flash_attn.__version__)

# Retrieve intitial conditions

In [ ]:
def get_open_data(param, levelist=[], **kwargs):
    fields = defaultdict(list)
    # Get the data for the current date and the previous date as the model is initialised with t-6h data
    for date in [DATE - datetime.timedelta(hours=6), DATE]:
        data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist, source = SOURCE, **kwargs)
        
        for f in data: # type: ignore
            # Open data is between -180 and 180, we need to shift it to 0-360
            assert f.to_numpy().shape == (721,1440)
            values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
            # Interpolate the data to from 0.25 to N320
            values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
            # Add the values to the list
            name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levelist else f.metadata("param")
            fields[name].append(values)
            
    # Create a single matrix for each parameter
    for param, values in fields.items():
        fields[param] = np.stack(values)

    return fields

PARAM_SFC = ["10u", "10v", "2d", "2t", "msl", "skt", "sp", "tcw", "lsm", "z", "slor", "sdor", "sd"]
PARAM_SOIL =["vsw","sot"]
PARAM_WAVE =["wmb", "h1012", "h1214", "h1417", "h1721", "h2125", "h2530", "mwd", "cdww", "mwp", "swh"]
PARAM_PL  = ["gh", "t", "u", "v", "q"]
LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50, 10]
SOIL_LEVELS = [1,2]

SOURCE = "ecmwf" # Other options are: "azure", "aws", "ecmwf" or "google" 

DATE = OpendataClient(SOURCE).latest()
print("Initial date is", DATE)

fields = {}

ekd.settings.set("cache-policy", "user")

fields.update(get_open_data(param=PARAM_SFC, levtype="sfc"))
assert all(p in fields for p in PARAM_SFC), "Missing parameters: %s" % (set(PARAM_SFC) - set(fields.keys()))

fields.update(get_open_data(param=PARAM_WAVE, stream="wave"))
assert all(p in fields for p in PARAM_WAVE), "Missing parameters: %s" % (set(PARAM_WAVE) - set(fields.keys()))

soil=get_open_data(param=PARAM_SOIL,levelist=SOIL_LEVELS)

soil_names = [f"{p}_{lev}" for p in PARAM_SOIL for lev in SOIL_LEVELS]
assert all(p in soil for p in soil_names), "Missing parameters: %s" % (set(soil_names) - set(soil.keys()))

fields.update(get_open_data(param=PARAM_PL, levelist=LEVELS))

PRESSURE_NAMES = [f"{p}_{lev}" for p in PARAM_PL for lev in LEVELS]
assert all(p in fields for p in PRESSURE_NAMES), "Missing parameters: %s" % (set(PRESSURE_NAMES) - set(fields.keys()))

if not os.path.exists("lsm.grib"):
    lsm_data = ekd.from_source("ecmwf-open-data", date=DATE, param="lsm", source=SOURCE)
    lsm_data.save("lsm.grib")

    

# Transform Data

In [ ]:
mwd = fields.pop("mwd")
mwd_rad = np.deg2rad(mwd)

fields["cos_mwd"] = np.cos(mwd_rad)
fields["sin_mwd"] = np.sin(mwd_rad)

mapping = {'sot_1': 'stl1', 'sot_2': 'stl2',
           'vsw_1': 'swvl1','vsw_2': 'swvl2'}
for k,v in soil.items():
    fields[mapping[k]]=v
    
fields.pop("q_10", None)  # Remove the 10hPa level for specific humidity, as it is not used in the model
fields.pop("q_50", None);  # Remove the 50hPa level for specific humidity, as it is not used as a prognostic in the model

lsm_field = ekd.from_source("file", 'lsm.grib')[0]
lsm_values = np.roll(lsm_field.to_numpy(), -lsm_field.shape[1] // 2, axis=1)
lsm_interpolated = ekr.interpolate(lsm_values, {"grid": (0.25, 0.25)}, {"grid": "N320"})
mask = np.equal(lsm_interpolated, 0)

fields["sd"][:, mask] = np.nan
fields["swvl1"][:, mask] = np.nan
fields["swvl2"][:,mask] = np.nan

# Transform GH to Z
for level in LEVELS:
    gh = fields.pop(f"gh_{level}")
    fields[f"z_{level}"] = gh * 9.80665
    

# Create Runner and Initial State

In [ ]:
input_state = dict(date=DATE, fields=fields)
checkpoint = {"huggingface":"ecmwf/aifs-single-2.0"}
runner = SimpleRunner(checkpoint)

# Run Model

In [ ]:

LEAD_TIME = 360
states = []

for state in runner.run(input_state=input_state, lead_time=LEAD_TIME):
    state_snapshot = copy.deepcopy(state)
    states.append(state_snapshot)
    print_state(state_snapshot)

# Save Model Run

In [ ]:
def save_model_run(states, params, output_dir):
    resolution = 0.25
    data_var_arrays = {}
    forecast_times = []

    for state in states:
        fields = state["fields"]
        for param in params:
            values_native = fields[param]

            grid_values, grid_info = regrid(
                data=values_native,
                in_grid={"grid": "N320"},
                out_grid={"grid": [resolution, resolution]},
                backend="precomputed",
            )

            grid_values = np.flipud(grid_values)
            grid_values = np.roll(
                grid_values,
                -(grid_values.shape[1] // 2),
                axis=1,
            )

            data_var_arrays.setdefault(param, []).append(grid_values)

        forecast_times.append(input_state["date"] + state["step"])

    grid_lat = np.linspace(
        -90,
        90,
        int(round(180 / resolution)) + 1,
    )
    grid_lon = np.linspace(
        -180,
        180 - resolution,
        int(round(360 / resolution)),
    )

    data_vars = {}
    for param, data_arrays in data_var_arrays.items():
        grid_values_all = np.stack(data_arrays, axis=0)

        assert grid_values_all.shape == (
            len(forecast_times),
            len(grid_lat),
            len(grid_lon),
        )
        
        data_vars[param] = (["time", "lat", "lon"], grid_values_all)

    ds_clean = xr.Dataset(
        data_vars=data_vars,
        coords={
            "time": np.asarray(forecast_times, dtype="datetime64[ns]"),
            "lat": grid_lat,
            "lon": grid_lon,
        },
        attrs={
            "initialization_date": str(input_state["date"]),
            "grid_resolution": f"{resolution} degree",
        },
    )

    ds_clean = ds_clean.chunk(
        {
            "time": -1,
            "lat": 180,
            "lon": 180,
        }
    )

    
    out_path = Path(output_dir) / "aifs_ecmwf_025.zarr"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    ds_clean.to_zarr(
        out_path,
        mode="w",
        consolidated=True,
    )

    print(
        "Zarr save completed:",
        ds_clean.sizes,
    )


Save the model run locally to a Zarr archive. 

In [ ]:
OUTPUT_DIR = Path("aifs_forecast")
save_model_run(states, ["2t"], OUTPUT_DIR)

Copy the model output to google drive

In [ ]:
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

destination_dir = "/content/drive/MyDrive/aifs_forecast"

if os.path.exists(destination_dir):
    shutil.rmtree(destination_dir)

shutil.copytree(str(OUTPUT_DIR), destination_dir)

# Plot Model Output
Plot the model output temperature for a given city

In [ ]:
def get_coordinates(location_name: str):
    geocode_url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": location_name,
        "count": 1,
        "language": "en",
        "format": "json"
    }
    
    response = requests.get(geocode_url, params=params)
    response.raise_for_status()
    data = response.json()
    
    if "results" not in data or not data["results"]:
        raise ValueError(f"Could not find coordinates for location: '{location_name}'")
        
    match = data["results"][0]
    return {
        "name": match.get("name"),
        "country": match.get("country"),
        "latitude": match.get("latitude"),
        "longitude": match.get("longitude"),
        "timezone": match.get("timezone")
    }

city_name = "Paris"
coordinates = get_coordinates(city_name)
lon, lat = coordinates["longitude"], coordinates["latitude"]

ds = xr.open_zarr("aifs_forecast/aifs_ecmwf_025.zarr")

temperature = ds["2t"].sel(lon=lon, lat=lat, method="nearest") - 273.15

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    temperature["time"].values,
    temperature.values,
    marker="o",
    markersize=3,
)

nearest_lat = float(temperature["lat"])
nearest_lon = float(temperature["lon"])

ax.set_title(
    f"Forecast 2-m temperature for {coordinates['name']} "
    f"({nearest_lat:.2f}°, {nearest_lon:.2f}°)"
)
ax.set_xlabel("Forecast time (UTC)")
ax.set_ylabel("Temperature (°C)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


Plot the model output temperature for a given region

In [ ]:

REGION = "global"

region_settings = {
    "global": {
        "extent": None,
        "figsize": (15, 7),
        "vmin": -35,
        "vmax": 45,
    },
    "europe": {
        "extent": [-25, 45, 34, 72],
        "figsize": (12, 8),
        "vmin": -35,
        "vmax": 45,
    },
    "germany": {
        "extent": [5, 16, 47, 56],
        "figsize": (10, 8),
        "vmin": -35,
        "vmax": 45,
    },
}

if REGION not in region_settings:
    raise ValueError(
        f"Unbekannte Region: {REGION}. "
        "Verfügbare Optionen: global, europe, germany"
    )

settings = region_settings[REGION]

ds = xr.open_zarr("aifs_forecast/aifs_ecmwf_025.zarr")
temperature = ds["2t"][0, :, :] -273.15  # Convert from Kelvin to Celsius

fig = plt.figure(figsize=settings["figsize"])
ax = plt.axes(projection=ccrs.PlateCarree())

temperature.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="nipy_spectral",
    vmin=settings["vmin"],
    vmax=settings["vmax"],
    cbar_kwargs={
        "label": "2-m temperature (°C)",
        "extend": "both",
    },
)

if settings["extent"] is None:
    ax.set_global()
else:
    ax.set_extent(settings["extent"], crs=ccrs.PlateCarree())

ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(
    cfeature.LAKES,
    edgecolor="gray",
    facecolor="none",
    linewidth=0.3,
)

gridlines = ax.gridlines(
    draw_labels=True,
    linewidth=0.4,
    alpha=0.5,
)
gridlines.top_labels = False
gridlines.right_labels = False

ax.set_title(
    f"AIFS 2-m temperature — {REGION.capitalize()}"
)

plt.tight_layout()
plt.show()

Generate a animation of the temperature for a given region for the model run

In [ ]:
ds = xr.open_zarr("aifs_forecast/aifs_ecmwf_025.zarr")
temperature_animation = (ds["2t"] - 273.15).load()

lons = temperature_animation["lon"].values
lats = temperature_animation["lat"].values
temperature_values = temperature_animation.values

fig = plt.figure(figsize=settings["figsize"])
ax = plt.axes(projection=ccrs.PlateCarree())

if settings["extent"] is None:
    ax.set_global()
else:
    ax.set_extent(settings["extent"], crs=ccrs.PlateCarree())

mesh = ax.pcolormesh(
    lons,
    lats,
    temperature_values[0],
    transform=ccrs.PlateCarree(),
    cmap="nipy_spectral",
    vmin=settings["vmin"],
    vmax=settings["vmax"],
    shading="auto",
)

cbar = fig.colorbar(
    mesh,
    ax=ax,
    orientation="vertical",
    pad=0.02,
    extend="both",
)
cbar.set_label("2-m temperature (°C)")

ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(
    cfeature.LAKES,
    edgecolor="gray",
    facecolor="none",
    linewidth=0.3,
)

gridlines = ax.gridlines(
    draw_labels=True,
    linewidth=0.4,
    alpha=0.5,
)
gridlines.top_labels = False
gridlines.right_labels = False

title = ax.set_title("")


def update(frame):
    mesh.set_array(temperature_values[frame].ravel())

    forecast_time = temperature_animation.time.values[frame]
    title.set_text(
        f"AIFS 2-m temperature — {REGION.capitalize()} — "
        f"{np.datetime_as_string(forecast_time, unit='h')}"
    )

    return mesh, title


animation = FuncAnimation(
    fig,
    update,
    frames=temperature_animation.sizes["time"],
    interval=1000,
    blit=False,
    repeat=True,
)

plt.close(fig)
display(HTML(animation.to_jshtml()))